# Lesson 5.2 — Risks, Failure Modes, and Human-in-the-Loop

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Enumerate LLM-specific risks and failure modes in a risk register
- Decide where human-in-the-loop (HITL) validation is required
- Route actions by impact and confidence to a human when warranted

**You will produce:** A risk register and a HITL routing policy keyed to impact and confidence.

## Concepts

LLM systems have characteristic **failure modes**: hallucination, prompt injection, data leakage, bias, over-confidence, stale knowledge, non-determinism, and cascading errors in agentic loops. A **risk register** records each risk with its likelihood, impact, and mitigation — the governance backbone reviewers expect.

**Human-in-the-loop (HITL)** inserts human judgment where automation is too risky. The architect decides *where*: high-impact/irreversible actions (refunds, medical decisions, account deletion) and low-confidence outputs should route to a human; low-impact/high-confidence ones can auto-proceed. The design pattern is an **impact x confidence** matrix driving auto / review / escalate decisions — tying together the escalation paths from Phase 1 and the confidence gates from Lesson 1.4.

> **Reference:** [Anthropic — Reducing hallucinations & oversight](https://docs.anthropic.com/en/docs/test-and-evaluate/strengthen-guardrails/reduce-hallucinations)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Build the risk register

We record risks with likelihood, impact, and mitigation. Scoring likelihood x impact prioritizes what to mitigate first.

In [ ]:
risks = [
  {"risk":"hallucinated coverage rule","likelihood":3,"impact":5,"mitigation":"RAG grounding + citation check + HITL"},
  {"risk":"prompt injection via fax text","likelihood":2,"impact":4,"mitigation":"input guardrail + least privilege"},
  {"risk":"PHI leakage in logs","likelihood":2,"impact":5,"mitigation":"redact before logging"},
  {"risk":"bias across member demographics","likelihood":3,"impact":4,"mitigation":"fairness eval slices (5.3)"},
]
for r in sorted(risks, key=lambda x: x['likelihood']*x['impact'], reverse=True):
    print(f"score {r['likelihood']*r['impact']:2}  {r['risk']:32} -> {r['mitigation']}")

## Code-Along: Step 2 — Encode the HITL routing policy

We route each decision by impact and model confidence. Irreversible/high-impact or low-confidence cases always get a human; routine confident cases auto-proceed.

In [ ]:
def hitl_route(impact, confidence, irreversible):
    if irreversible or impact >= 4:
        return "human_review_required"
    if confidence < 0.7:
        return "human_review_required"
    if confidence < 0.85:
        return "human_spot_check"
    return "auto_proceed"

for imp, conf, irr in [(5,0.95,True),(2,0.6,False),(2,0.8,False),(1,0.99,False)]:
    print(f"impact={imp} conf={conf} irreversible={irr} -> {hitl_route(imp,conf,irr)}")

## Exercise

Add a **cost-of-review** consideration: if review volume would exceed human capacity, the policy should raise the auto-proceed confidence threshold for *low-impact* cases only (never for high-impact). Implement a capacity-aware variant and show how the auto/review mix shifts as capacity tightens — this is the real-world SLA vs safety trade-off.

In [ ]:
# TODO: capacity-aware hitl_route that tunes low-impact thresholds under load
# TODO: simulate a batch and report the auto vs review mix at two capacity levels

## Key Takeaways

- Maintain a risk register scored by likelihood x impact with explicit mitigations.
- Route high-impact/irreversible or low-confidence outputs to humans.
- HITL is an impact x confidence decision, not an all-or-nothing switch.
- Review capacity is finite — balance safety against SLA deliberately.

---

**Next:** [Lesson 5.3 — Compliance and Ethical AI](./lesson-5.3-compliance-ethical-ai.ipynb)